In [ ]:
import random
import asyncio
from bleak import BleakClient, BleakScanner

# ── Config ──────────────────────────────────────────────
DEVICE_NAME = "Adafruit Bluefruit LE"  # change if yours is named differently
UART_RX_UUID = "6e400003-b5a3-f393-e0a9-e50e24dcca9e"  # Bluefruit UART notify

# ── Game State ───────────────────────────────────────────
score = 0
level = 1

# ── FSM ─────────────────────────────────────────────────
class BopItFSM:
    def __init__(self):
        self.current_state = "idle"
        self.current_prompt = None

    def prompt(self):
        self.current_prompt = random.choice(["ATTACK", "BLOCK"])
        self.current_state = "waiting"
        print(f"\n>>> {self.current_prompt}!")

    def evaluate(self, detected: str):
        global score
        if self.current_state != "waiting":
            return

        if detected == self.current_prompt:
            print("✓ Correct!")
            score += 1
        else:
            print(f"✗ Wrong! Expected {self.current_prompt}, got {detected}")
        
        print(f"Score: {score}")
        self.current_state = "idle"

    def reset(self):
        global score, level
        score = 0
        level = 1
        self.current_state = "idle"
        print("Game reset.")

# ── BLE Data Handler ─────────────────────────────────────
fsm = BopItFSM()

def on_data(sender, data: bytearray):
    message = data.decode("utf-8").strip().upper()
    print(f"[BLE] Received: '{message}'")

    if message in ("ATTACK", "BLOCK"):
        fsm.evaluate(message)
        # Wait 1 second then prompt next round
        asyncio.get_event_loop().call_later(1.0, fsm.prompt)
    # If empty or anything else, ignore it

# ── Main ─────────────────────────────────────────────────
async def main():
    print("Scanning for Bluefruit...")
    device = await BleakScanner.find_device_by_name(DEVICE_NAME)
    if not device:
        print(f"Could not find '{DEVICE_NAME}'. Is it powered on?")
        return

    print(f"Found {device.name}, connecting...")

    async with BleakClient(device) as client:
        print("Connected! Starting game...\n")
        await client.start_notify(UART_RX_UUID, on_data)
        
        fsm.prompt()  # kick off first round

        try:
            while True:
                await asyncio.sleep(1)
        except KeyboardInterrupt:
            print("\nGame over!")
            print(f"Final score: {score}")
            await client.stop_notify(UART_RX_UUID)

asyncio.run(main())